# HamSearch

### Setup

Import dependencies and initialize the Persian normalizer and lemmatizer.


In [ ]:
import math
import os
import re

import numpy as np
import pandas as pd
from hazm import Normalizer, word_tokenize, Lemmatizer

normalizer = Normalizer()
lemmatizer = Lemmatizer()


### Load Data

Load documents, queries, relevance judgments, and stopwords. Validate paths and IDs, and sort documents consistently.


In [ ]:
CORPUS_DIR  = "HamshahriData/HamshahriCorpus"
QUERIES_DIR = "HamshahriData/Queries"
QRELS_PATH  = "HamshahriData/RelativeAssesemnt/judgements.txt"
STOPWORDS_PATH = "HamshahriData/persian_stopwords.txt"

def load_hamshahri_dataset():
    docs = []

    for root, dirs, files in os.walk(CORPUS_DIR):
        dirs.sort()
        for fname in sorted(files):
            if not fname.endswith(".ham"):
                continue

            fpath = os.path.join(root, fname)
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read().strip()

            if not raw:
                continue

            lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
            if not lines:
                continue

            docs.append({
                "doc_id": os.path.splitext(fname)[0],
                "title": lines[0],
                "text": " ".join(lines[1:]) if len(lines) > 1 else ""
            })

    if not docs:
        raise ValueError(f"No non-empty .ham documents found in {CORPUS_DIR}")
    docs_df = pd.DataFrame(docs).sort_values("doc_id").reset_index(drop=True)
    if docs_df["doc_id"].duplicated().any():
        raise ValueError("Duplicate document IDs found in the corpus.")

    queries = []

    for root, dirs, files in os.walk(QUERIES_DIR):
        dirs.sort()
        for fname in sorted(files):
            if not fname.endswith(".q"):
                continue

            fpath = os.path.join(root, fname)
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                qtext = f.read().strip()

            queries.append({
                "query_id": os.path.splitext(fname)[0],  
                "query_text": qtext
            })

    if not queries:
        raise ValueError(f"No .q query files found in {QUERIES_DIR}")

    queries_df = (
        pd.DataFrame(queries)
        .sort_values("query_id", key=lambda s: s.astype(int))
        .reset_index(drop=True)
    )

    qrels = {}
    with open(QRELS_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                qid, docid = parts[0], parts[1]
                if qid not in qrels:
                    qrels[qid] = set()
                qrels[qid].add(docid)

    if queries_df["query_id"].duplicated().any():
        raise ValueError("Duplicate query IDs found in the dataset.")
    if set(queries_df["query_id"]) != set(qrels):
        raise ValueError("Query IDs and relevance-judgment IDs do not match.")
    missing_docs = set().union(*qrels.values()) - set(docs_df["doc_id"])
    if missing_docs:
        raise ValueError(f"Relevance judgments reference {len(missing_docs)} missing documents.")
    return docs_df, queries_df, qrels

for directory in (CORPUS_DIR, QUERIES_DIR):
    if not os.path.isdir(directory):
        raise FileNotFoundError(
            f"Dataset folder not found: {directory}. "
            "Run this notebook from the folder containing HamshahriData."
        )
for filepath in (QRELS_PATH, STOPWORDS_PATH):
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"Required dataset file not found: {filepath}")

with open(STOPWORDS_PATH, "r", encoding="utf-8-sig") as f:
    raw_stopwords = set([line.strip() for line in f if line.strip()])
stopwords = set(normalizer.normalize(w) for w in raw_stopwords)

docs_df, queries_df, qrels = load_hamshahri_dataset()

docs_df.shape, queries_df.shape, len(qrels)


### Preprocess Text

Normalize and tokenize text, remove digits and stopwords, then lemmatize documents and queries using the same pipeline.


In [ ]:
def preprocess(text, return_tokens=False, remove_digits=True):
    if text is None:
        text = ""
    text = str(text)

    text = normalizer.normalize(text)
    text = text.replace("\u200c", " ")  

    if remove_digits:
        text = re.sub(r"[0-9۰-۹]+", " ", text)

    text = re.sub(r"[^A-Za-z\u0600-\u06FF\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return [] if return_tokens else ""

    tokens = word_tokenize(text)

    tokens = [t for t in tokens if (t not in stopwords and len(t) > 1)]

    tokens = [lemmatizer.lemmatize(t).split("#")[0] for t in tokens]

    return tokens if return_tokens else " ".join(tokens)
print("Sample doc title:", docs_df.loc[0, "title"])
print("Before:", docs_df.loc[0, "text"][:200], "...")
print("After (lemma):", preprocess(docs_df.loc[0, "text"])[:200], "...")

docs_df["doc_clean"] = (docs_df["title"].fillna("") + " " + docs_df["text"].fillna("")).apply(preprocess)
queries_df["query_clean"] = queries_df["query_text"].fillna("").apply(preprocess)
docs_df[["doc_id","doc_clean"]].head(2), queries_df[["query_id","query_clean"]].head(2)


### Token Lists

Split cleaned text into token lists and count the documents.


In [ ]:
def to_tokens(s):
    return s.split() if s else []

docs_tokens = [to_tokens(x) for x in docs_df["doc_clean"].tolist()]
queries_tokens = [to_tokens(x) for x in queries_df["query_clean"].tolist()]
N = len(docs_tokens)


### Vocabulary

Assign consistent term IDs and count how many documents contain each term.


In [ ]:
term2number = {}
df = []  

for tokens in docs_tokens:
    seen = set(tokens)
    for t in sorted(seen):
        if t not in term2number:
            term2number[t] = len(term2number)
            df.append(0)
        df[term2number[t]] += 1

V = len(term2number)
print("Num docs:", N)
print("Vocab size:", V)


### Inverse Document Frequency

Compute smoothed IDF weights: log((N + 1) / (df + 1)) + 1.


In [ ]:
# df[i] counts documents containing term i.
df = np.array(df, dtype=np.int32)
idf = np.log((N + 1) / (df + 1)) + 1.0

### TF-IDF Vectors

Build sparse document vectors using term frequency divided by document length, multiplied by IDF.


In [ ]:
def build_tfidf_vector(tokens, term2id, idf):
    if not tokens:
        return {}
    counts = {}
    for t in tokens:
        tid = term2id.get(t)
        if tid is None:
            continue
        counts[tid] = counts.get(tid, 0) + 1

    L = len(tokens)
    vec = {}
    for tid, c in counts.items():
        tf = c / L
        vec[tid] = tf * float(idf[tid])
    return vec

doc_vecs = [build_tfidf_vector(toks, term2number, idf) for toks in docs_tokens]


### Vector Norms

Compute document vector lengths for cosine similarity.


In [ ]:
def vec_norm(vec):
    s = 0.0
    for w in vec.values():
        s += w * w
    return math.sqrt(s)

doc_norms = np.array([vec_norm(v) for v in doc_vecs], dtype=np.float32)

### Rank Documents

Rank positive cosine matches by score, breaking ties by document ID. Return no matches for empty or unknown queries.


In [ ]:
def cosine_score_sparse(doc_vec, doc_norm, q_vec, q_norm):
    if doc_norm == 0 or q_norm == 0:
        return 0.0
    dot = 0.0
    for tid, qw in q_vec.items():
        dw = doc_vec.get(tid)
        if dw is not None:
            dot += dw * qw
    return dot / (doc_norm * q_norm)


def validate_k(k):
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 0:
        raise ValueError("k must be a non-negative integer.")


def rank_documents_for_query_tokens(q_tokens, top_k=10):
    validate_k(top_k)
    empty_result = pd.DataFrame({
        "doc_id": pd.Series(dtype="object"),
        "title": pd.Series(dtype="object"),
        "score": pd.Series(dtype="float32"),
    })
    if top_k == 0:
        return empty_result

    q_vec = build_tfidf_vector(q_tokens, term2number, idf)
    q_norm = vec_norm(q_vec)
    if q_norm == 0:
        return empty_result

    scores = np.zeros(N, dtype=np.float32)
    for i in range(N):
        scores[i] = cosine_score_sparse(
            doc_vecs[i], float(doc_norms[i]), q_vec, q_norm
        )

    # Return only positive matches; break score ties by document ID.
    candidates = np.flatnonzero(scores > 0)
    if candidates.size == 0:
        return empty_result
    candidate_ids = docs_df.iloc[candidates]["doc_id"].to_numpy(dtype=str)
    order = np.lexsort((candidate_ids, -scores[candidates]))
    top_idx = candidates[order[:top_k]]
    res = docs_df.iloc[top_idx][["doc_id", "title"]].copy()
    res["score"] = scores[top_idx]
    return res.reset_index(drop=True)


### Example Query

Retrieve the top 10 matches for the query at row index 20 (query ID 21).


In [ ]:
# Zero-based row index: 20 selects query ID 21 in this dataset.
qi = 20
print("Query ID:", queries_df.loc[qi, "query_id"])
print("Query:", queries_df.loc[qi, "query_text"])
rank_documents_for_query_tokens(queries_tokens[qi], top_k=10)


### Evaluation Metrics

Define Precision@K, Recall@K, F1@K, AP@K, and MAP@K. Precision divides by K; AP@K divides by min(R, K), while untruncated AP divides by R (the number of relevant documents).


In [ ]:
def precision_at_k(ranked_docs, relevant_set, k):
    validate_k(k)
    if k == 0:
        return 0.0
    rel_retrieved = sum(d in relevant_set for d in ranked_docs[:k])
    return rel_retrieved / k


def recall_at_k(ranked_docs, relevant_set, k):
    validate_k(k)
    if not relevant_set:
        return 0.0
    rel_retrieved = sum(d in relevant_set for d in ranked_docs[:k])
    return rel_retrieved / len(relevant_set)


def f1_at_k(ranked_docs, relevant_set, k):
    p = precision_at_k(ranked_docs, relevant_set, k)
    r = recall_at_k(ranked_docs, relevant_set, k)
    return 2 * p * r / (p + r) if p + r else 0.0


def average_precision(ranked_docs, relevant_set, k=None):
    """AP@k uses min(R, k); untruncated AP uses R relevant documents.

    ranked_docs must contain unique document IDs, as returned by the ranker.
    """
    if k is not None:
        validate_k(k)
    if not relevant_set:
        return 0.0

    cutoff = len(ranked_docs) if k is None else k
    denom = len(relevant_set) if k is None else min(len(relevant_set), k)
    if denom == 0:
        return 0.0

    hit = 0
    sum_precisions = 0.0
    for i, doc_id in enumerate(ranked_docs[:cutoff], start=1):
        if doc_id in relevant_set:
            hit += 1
            sum_precisions += hit / i
    return sum_precisions / denom


def mean_average_precision(results, qrels, k=None):
    if k is not None:
        validate_k(k)
    aps = [
        average_precision(results.get(str(qid), []), relset, k=k)
        for qid, relset in qrels.items()
    ]
    return float(np.mean(aps)) if aps else 0.0


### Retrieve All Queries

Retrieve up to TOP_K matching documents for each query and store their IDs.


In [ ]:
TOP_K = 10
results = {}

for i in range(len(queries_tokens)):
    qid = str(queries_df.loc[i, "query_id"])
    ranked_df = rank_documents_for_query_tokens(queries_tokens[i], top_k=TOP_K)
    results[qid] = ranked_df["doc_id"].tolist()


### Overall Results

Report mean Precision, Recall, F1, and AP across queries at the same retrieval depth.


In [ ]:
K = TOP_K 

precisions = []
recalls = []
f1s = []

for qid, relset in qrels.items():
    ranked = results.get(str(qid), [])
    precisions.append(precision_at_k(ranked, relset, K))
    recalls.append(recall_at_k(ranked, relset, K))
    f1s.append(f1_at_k(ranked, relset, K))

P_at_K = float(np.mean(precisions)) if precisions else 0.0
R_at_K = float(np.mean(recalls)) if recalls else 0.0
F1_at_K = float(np.mean(f1s)) if f1s else 0.0
MAP_at_K = mean_average_precision(results, qrels, k=K)

print(f"Avg Precision@{K}: {P_at_K:.4f}")
print(f"Avg Recall@{K}:    {R_at_K:.4f}")
print(f"Avg F1@{K}:        {F1_at_K:.4f}")
print(f"MAP@{K}:           {MAP_at_K:.4f}")


### Per-Query Results

Show the five highest- and lowest-scoring queries by AP@K, including each query’s relevant-document count.


In [ ]:
def evaluate_per_query(results, qrels, k=10, n_show=5):
    rows = []
    for qid, relset in qrels.items():
        ranked = results.get(str(qid), [])
        rows.append({
            "query_id": str(qid),
            f"P@{k}": precision_at_k(ranked, relset, k),
            f"R@{k}": recall_at_k(ranked, relset, k),
            f"F1@{k}": f1_at_k(ranked, relset, k),
            f"AP@{k}": average_precision(ranked, relset, k=k),
            "num_rel": len(relset)
        })
    df = pd.DataFrame(rows).sort_values(f"AP@{k}", ascending=False, kind="stable").reset_index(drop=True)
    return df.head(n_show), df.tail(n_show)

top5, bottom5 = evaluate_per_query(results, qrels, k=K, n_show=5)
top5, bottom5
